# Introduction: Primer Design for LAMP ML

This notebook is for designing primers for LAMP ML dataset 
Target design is in a separate notebook 

Last updated: 2025-04-09 by ZCL 

In [ ]:
#import required packages 
import os

#dfs and plotting 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#biological seq data 
from Bio import AlignIO
from Bio import SeqIO
from Bio.Seq import Seq

#primer characteristics 
import primer3
from Bio.SeqUtils import gc_fraction
import primer3.bindings as p3b

#misc 
import itertools
import datetime
import random 
from collections import Counter

import matplotlib.patches as mpatches 

from datetime import date

# Define class 

In [ ]:
class LampPrimerGeneration:
    
    def __init__(self, target_sequence, length_rules):
        """Initialize with raw fluorescence and plate layout inputs.
        
        Args:
           target_sequence: target sequence to design against 
           length_rules: dict with min and max length for each primer type 
           spacing rules : dict with min and max lenght for each inter-primer space 
            
            
        """
        self.target_seq = target_sequence
        self.length_rules = length_rules
        
    def generate_tiling_sets(self, offset = 0, frame_size = 8, gaps = [5,0,0,5,0,0,5,0]):
        
       
        start_position = offset
        end_position = len(self.target_seq)  

        indiv_primers = {} 
        self.all_sets = []
         
        stop_outer_loop = False
        
        global_primer_counter = 0 

        #loop through set start position (ie F3 start) 
        for n,set_start in  enumerate(list(range(start_position,end_position,frame_size))) : 
            if stop_outer_loop:
                break
                
            pset = []
            
            primer_start = set_start #set the primer start to the set start for the first primer 
            
            for ptype,gap in zip(["F3","F2","FL","F1c","B1c","BL","B2","B3"], gaps): 

                min_val = self.length_rules[ptype].get('min')
                max_val = self.length_rules[ptype].get('max')

                mean = (min_val + max_val) / 2
                    # Setting std dev so that most values (about 99.7%) fall within our range
                std_dev = (max_val - min_val) / 6

                length = int(np.random.normal(mean, std_dev))
    
                primer_end = primer_start + length
                seq = self.target_seq[primer_start : primer_end] 

                #check there's enough sequence to make the desired length primer
                if primer_end > len(self.target_seq) :
                    print(f"Not enough target sequence length to generate primer set {n}")
                    stop_outer_loop = True
                    break 
                    
                else : 
                    pset.append(f"primer-{global_primer_counter}") #add to pset 
                    
                    if ptype in ["F3","F2","B1c","BL"] : 
                        indiv_primers[f"primer-{global_primer_counter}"] = {'primer_set':n,'primer_type':ptype,'sequence':seq,'start':primer_start,'end':primer_start+ length,'direction':'F'}
                    else :
                        indiv_primers[f"primer-{global_primer_counter}"] = {'primer_set':n,'primer_type':ptype,'sequence':str(Seq(seq).reverse_complement()),'start':primer_start,'end':primer_start+ length,'direction':'RC'}
                        
                    global_primer_counter +=1 #add to primer count
                    primer_start = primer_end + gap #set the start for the next primer in the set
            
            if len(pset) == 8 :
                self.all_sets.append(pset)
                    
        self.all_primers = {} 
        
        for ptype in ["F3","F2","FL","F1c","B1c","BL","B2","B3"]: 
            self.all_primers[ptype] = {name: details for name, details in indiv_primers.items() if details['primer_type'] == ptype}

        print(f"Generated {len(self.all_sets)} tiling primer sets ")
            
        return self.all_primers, self.all_sets


    
    def filter_possible_primers(self, my_filters, 
        gc_range=(0.2, 0.8), 
        homodimer_threshold=-9000,
        tm_ranges=  None, 
        oligo_conc=100, 
        na_conc=50, 
        mg_conc=4):
        """
        Filters primers based on specified criteria. Primers must pass all specified filters.

        Parameters:
            all_primers (dict): A nested dictionary of primers. Expected format:
                {ptype: {primer: {'sequence': sequence_str}}}
            my_filters (list): List of filters to apply. Supported filters: ["gc", "homodimer_dg", "tm"].
            gc_range (tuple): Min and max GC fraction (inclusive) for filtering.
            homodimer_threshold (float): Threshold for acceptable homodimer delta G.
            tm_ranges (dict): Dictionary specifying Tm ranges for each ptype. Example:
                {
                    "F1c": (64, 66),
                    "B1c": (64, 66),
                    "Loop": (64, 66),
                    "F2": (59, 61),
                    "B2": (59, 61),
                    "F3": (59, 61),
                    "B3": (59, 61),
                }
            oligo_conc (float): Oligonucleotide concentration (µM) for Tm calculation.
            na_conc (float): Sodium ion concentration (mM) for Tm calculation.
            mg_conc (float): Magnesium ion concentration (mM) for Tm calculation.

        Returns:
            list: A list of primer names that satisfy all specified filters.
        """

        # Use default Tm ranges if not provided
        if tm_ranges is None:
            tm_ranges = {
                "F1c": (64, 66),
                "B1c": (64, 66),
                "FL": (64, 66),
                "BL": (64, 66),
                "F2": (59, 61),
                "B2": (59, 61),
                "F3": (59, 61),
                "B3": (59, 61),
            }
        
        # Supported filters
        possible_filters = {"gc", "homodimer_dg","tm"}

        # Validate filters
        invalid_filters = set(my_filters) - possible_filters
        if invalid_filters:
            raise ValueError(f"Invalid filter options: {', '.join(invalid_filters)}")

        # Initialize a list to collect primers that pass all filters
        self.filtered_primers = []

        # Iterate through all primers
        for ptype, primers in self.all_primers.items():
            for primer, val in primers.items():
                sequence = val.get('sequence')

                # Check GC filter
                if "gc" in my_filters:
                    gc = gc_fraction(sequence)
                    if not (gc_range[0] <= gc <= gc_range[1]):
                        continue  # Skip this primer if it fails the GC filter

                # Check homodimer delta G filter
                if "homodimer_dg" in my_filters:
                    homodimer_dg = p3b.calc_homodimer(sequence).dg
                    if homodimer_dg < homodimer_threshold:
                        continue  # Skip this primer if it fails the homodimer filter

                # Check tm filter
                if "tm" in my_filters:

                    tm = p3b.calc_tm(val.get('sequence'), mv_conc=na_conc, dv_conc=mg_conc, dna_conc=oligo_conc)

                    tm_range = tm_ranges.get(ptype)
                    if tm_range and not (tm_range[0] <= tm <= tm_range[1]):
                        continue  # Skip this primer if Tm is outside the specified range


                # If the primer passes all filters, add it to the list
                self.filtered_primers.append(primer)

        # Filter the dictionary
        self.filtered_dict = {
            group: {name: details for name, details in self.all_primers[group].items() if name in self.filtered_primers}
            for group in self.all_primers
        }

        for ptype in ["F3","FL","F2","F1c","B3","BL","B2","B1c"] :
        
            print(f"{len(self.filtered_dict[ptype])} / {len(self.all_primers[ptype])} ({100* len(self.filtered_dict[ptype]) /len(self.all_primers[ptype]):.1f}%) of\
            {ptype} primers retained after filtering for {my_filters}")
            
        #Filter the sets
        self.potential_sets = [pset for pset in self.all_sets if set(pset).issubset(self.filtered_primers)]
                               
        return self.potential_sets
        
        
    def plot_sets_per_block(self, n_blocks, len_block, palette = "tab20") :
        
        #if you didn't do any filtering then define potential sets as all sets 
        if not hasattr(self, 'potential_sets'):
            self.potential_sets = self.all_sets
            
        block_bounds = {f'block_{i}':(i*len_block, i*len_block + len_block) for i in range(n_blocks)}
        block_colors = {f'block_{i}': color for i, color in enumerate(sns.color_palette(palette, n_blocks))}

        #sort based on which "block" of the target they fall in to 
        self.n_blocks = n_blocks 
        self.len_block = len_block 
        self.block_psets = {}
        for block in block_bounds.keys() :
            brange = block_bounds[block]
            within = []
            transition = [] 

            for pset in self.potential_sets : 
                F3 = pset[0]
                start = self.all_primers['F3'][F3].get('start')
                if (start >= brange[0]) & (start < brange[1]) : #if the start is in the range then look at the end 

                    B3 = pset[7]
                    end = self.all_primers['B3'][B3].get('end')

                    if (end >= brange[0]) & (end < brange[1]) :
                        within.append(pset) 
                    elif end > brange[1] : 
                        transition.append(pset) 
                    else :
                        pass#print("Error finding end!")
                else : 
                    pass 

            self.block_psets[block] = within
            self.block_psets[f"{block}_transition"] = transition

        # #plot
        plt.bar(x = self.block_psets.keys(), height = [len(self.block_psets[key]) for key in self.block_psets.keys()],
               color = [block_colors.get(key,'lightgrey') for key in self.block_psets.keys() ])
        plt.title(f"Primer sets per region")
        plt.xlabel("Block of target sequence") 
        plt.ylabel("Number of primer sets") 
        plt.xticks(ticks = range(len(self.block_psets.keys())), labels = self.block_psets.keys(), rotation = 45, ha = 'right');
        
        return self.block_psets
    
    def downselect_and_limit_psets_per_transition_block(self,n_max_block, n_max_transition, palette = 'tab20'):
        block_bounds = {f'block_{i}':(i*self.len_block, i*self.len_block + self.len_block) for i in range(self.n_blocks)}
        block_colors = {f'block_{i}': color for i, color in enumerate(sns.color_palette(palette, len(self.block_psets.keys())))}

        #limit to n per block / transition 
        self.selected_psets = []
        for key, psets in self.block_psets.items() :
            if 'transition' in key :
                if n_max_transition < len(psets) :
                    self.selected_psets += random.sample(psets, n_max_transition)
                else :
                    self.selected_psets += psets
            else :
                if n_max_block < len(psets) :
                    self.selected_psets += random.sample(psets, n_max_block)
                else :
                    self.selected_psets += psets
            
                
        #recalc and plot psets per block 
        #sort based on which "block" of the target they fall in to 
        self.block_psets = {}
        for block in block_bounds.keys() :
            brange = block_bounds[block]
            within = []
            transition = [] 

            for pset in self.selected_psets : 
                F3 = pset[0]
                start = self.all_primers['F3'][F3].get('start')
                if (start >= brange[0]) & (start < brange[1]) : #if the start is in the range then look at the end 

                    B3 = pset[7]
                    end = self.all_primers['B3'][B3].get('end')

                    if (end >= brange[0]) & (end < brange[1]) :
                        within.append(pset) 
                    elif end > brange[1] : 
                        transition.append(pset) 
                    else :
                        pass#print("Error finding end!")
                else : 
                    pass 

            self.block_psets[block] = within
            self.block_psets[f"{block}_transition"] = transition

        # #plot
        plt.bar(x = self.block_psets.keys(), height = [len(self.block_psets[key]) for key in self.block_psets.keys()], 
               color = [block_colors.get(key,'lightgrey') for key in self.block_psets.keys() ])
        plt.title(f"Primer sets per region")
        plt.xlabel("Block of target sequence after restrictions") 
        plt.ylabel("Number of primer sets") 
        plt.xticks(ticks = range(len(self.block_psets.keys())), labels = self.block_psets.keys(), rotation = 45, ha = 'right');
        
        #change the potential sets to the filtered sets 
        print(f"After restriction to {n_max_block} primer sets per block and {n_max_transition} per transition,the total number of psets is {len(self.selected_psets)} from {len(self.potential_sets)}")
        self.potential_sets = self.selected_psets
        

    def plot_amplicon_starts(self, bin_size = 20) :
        starts = []
        for pset in self.selected_psets : 
            F3 = pset[0]
            starts.append(self.all_primers['F3'][F3].get('start'))
            
        plt.hist(starts, bins = np.arange(0,len(self.target_seq),bin_size))
        plt.title(f"Amplicon start position")
        plt.xlabel("Position in target sequence") 
        plt.ylabel("Number of primer sets") 
        
    def calc_primer_usage(self):
        print(len(self.potential_sets))
        total_primers = 0
        for i, ptype in enumerate(["F3","FL","F2","F1c","B3","BL","B2","B1c"]) : 

            ptype_list = []
            for item in self.potential_sets : 
                ptype_list.append(item[i])

            if ptype not in ["F2","F1c","B2","B1c"] : 
                total_primers += len(set(ptype_list) )

                print(f"{len(set(ptype_list))} unique {ptype} primers out of {len(self.source_primers[ptype])} used in in all possible sets ") 

        #how many FIP and BIP would we need to order? 
        unique_FIP = set([pset[3] + "-" + pset[1] for pset in self.potential_sets])
        unique_BIP = set([pset[4] + "-" + pset[6] for pset in self.potential_sets])

        total_primers = total_primers + len(unique_FIP) + len(unique_BIP) 

        print(f"{len(unique_FIP)} unique FIP used in all possible sets")
        print(f"{len(unique_BIP)} unique BIP used in all possible sets")

        print(f"total number of unique primers to order = {total_primers:,} ie {np.ceil(total_primers/96):.0f} x 96 well plates")
    
    def plot_uses_per_primer(self) :# how many times is each primer used? 

        fig,axes = plt.subplots(1,6, figsize = (10,2)) 
        fig.tight_layout(w_pad = 2)
        j = 0
        for i, ptype in enumerate(["F3","FL","F2","F1c","B3","BL","B2","B1c"]) : 

            if ptype not in ["F2","F1c","B2","B1c"]: 
                ptype_list = []
                for item in self.potential_sets : 
                    ptype_list.append(item[i])

                counts = list(Counter(ptype_list).values())

                axes[j].hist(counts)
                axes[j].set_title(ptype)
                j+= 1
                print(f"The most frequent {ptype} primer is used {np.max(counts)} times; the least frequent is used {np.min(counts)} times") 

        counts_FIP = list(Counter([pset[3] + "-" + pset[1] for pset in self.potential_sets]).values())
        print(f"The most frequent FIP primer is used {np.max(counts_FIP)} times; the least frequent is used {np.min(counts_FIP)} times") 

        counts_BIP = list(Counter([pset[4] + "-" + pset[6] for pset in self.potential_sets]).values())
        print(f"The most frequent BIP primer is used {np.max(counts_BIP)} times; the least frequent is used {np.min(counts_BIP)} times") 

        axes[j].hist(list(Counter([pset[3] + "-" + pset[1] for pset in self.potential_sets]).values()))
        axes[j].set_title("FIP")
        axes[j+1].hist(list(Counter([pset[4] + "-" + pset[6] for pset in self.potential_sets]).values()))
        axes[j+1].set_title("BIP")

        for ax in axes :
            ax.set_ylabel("Number of primers")
            ax.set_xlabel("Number of uses ")
            
    def visualize_binding_sites(self, limit = 50):
        
        if limit > len(self.potential_sets) :
            limit = len(self.potential_sets) 
        
        fig,ax= plt.subplots(1,1,figsize = (14,6))

        ax.hlines(y=0, xmin =0, xmax = len(target_seq), color = "black", linewidth = 6)

        # Create a Tab20 color palette
        colors = plt.cm.tab20.colors

        # Adjusted loop
        for i in range(limit):  #
            pset = random.choice(self.potential_sets)
            for primer,ptype,color in zip(pset,["F3","F2","FL","F1c","B1c","BL","B2","B3"],colors[:8]) :
                ax.hlines(y=i+1, xmin = self.all_primers[ptype][primer].get('start'), 
                           xmax= self.all_primers[ptype][primer].get('end'), linewidth = 4, color = color)

        handles = [] 

        for ptype,color in zip(["target_seq","F3","F2","FL","F1c","B1c","BL","B2","B3"],["black"] + list((colors[:8]))): 
            handles.append(mpatches.Patch(color=color, label=ptype))

        fig.legend(handles = handles,title = "Primer type", bbox_to_anchor = (1.01,0.9))

        fig.suptitle(f"Subset of primer sets generated, n = {limit}", fontsize = 18)
        
    def generate_primer_dfs(self):
        #generate the primer bind df 
        filt_primers = [item for sublist in self.potential_sets for item in sublist]
        
        flattened_dict = {}
        for key, nested_dict in self.all_primers.items():
            flattened_dict.update({p: nested_dict[p] for p in nested_dict if p in filt_primers})

        self.primer_bind_df = pd.DataFrame.from_dict(flattened_dict, orient = 'index')
        
        #generate the primer ordering df 
        # NOTE -- currently missing adding the well number for the plate because still trying to decide exactly how we will order 
        dfs = []
        for i, ptype in enumerate(["F3","F2","FL","F1c","B1c","BL","B2","B3"]) : 
            if ptype not in ["F2","F1c","B2","B1c"]: 
                ptype_primers = [item[i] for item in self.potential_sets]
                #ptype_dict[ptype] = 
                
                df = pd.DataFrame.from_dict({key: self.all_primers[ptype][key] for key in ptype_primers if key in self.all_primers[ptype]}, orient = 'index')
                df['primer_type'] = ptype
                dfs.append(df)
            
        F2s = [item[1] for item in self.potential_sets]
        F1cs = [item[3] for item in self.potential_sets]
        FIP_dict = {} 
        for F1c, F2 in zip(F1cs,F2s):
            FIP_dict[f"{F1c}-{F2}"] = {'sequence': self.all_primers['F1c'][F1c].get('sequence')  + self.all_primers['F2'][F2].get('sequence'),
                                      'primer_type':'FIP'} 
        
        dfs.append(pd.DataFrame.from_dict(FIP_dict, orient = 'index'))
        
        B2s = [item[6] for item in self.potential_sets]
        B1cs = [item[4] for item in self.potential_sets]
        BIP_dict = {}
        for B1c, B2 in zip(B1cs,B2s):
            BIP_dict[f"{B1c}-{B2}"] = {'sequence': self.all_primers['B1c'][B1c].get('sequence')  + self.all_primers['B2'][B2].get('sequence'),
                                      'primer_type':'BIP'} 
        
        dfs.append(pd.DataFrame.from_dict(BIP_dict, orient = 'index'))
        
        self.primer_order_df = pd.concat(dfs)
        return self.primer_bind_df, self.primer_order_df
    


# Import target sequence to design against 

In [ ]:
#define some useful sequeneces 

twist_5p_adapter = 'CAATCCGCCCTCACTACAACCG'
twist_3p_adapter = 'CTACTCTGGCGTCGATGAGGGA'

T7 = 'TAATACGACTCACTATAG'

In [ ]:

target_seq = pd.read_csv("/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/Final_designs_3-13/WT_target_sequence_10blocks_350len_SR1.7_2025-03-13.csv",index_col = 0).sequence[0]
    
print(f"Target sequence selected and saved to variable target_seq. Lenth = {len(target_seq)}, GC = {gc_fraction(target_seq)}")


if target_seq.find(T7) == -1 :
    print("No T7 in seq")
else : 
    print(f"T7 sequence found at position {target_seq.find(T7) + 1}")
    
if target_seq.find(twist_5p_adapter) == -1 :
    print("No twist 5p adapter in seq")
else : 
    print(f"Twistp 5p adapter sequence found at position {target_seq.find(T7) + 1}")

# Generate primers 

In [ ]:
#define length rules 
# length_rules = {'F1c':{'min':20,'max':22},
#                'B1c':{'min':20,'max':22},
#                 'F2':{'min':18,'max':20},
#                 'B2':{'min':18,'max':20},
#                'F3':{'min':18,'max':20},
#                 'B3':{'min':18,'max':20},
#                 'FL':{'min':15,'max':25},
#                 'BL':{'min':15,'max':25}}

length_rules = {'F1c':{'min':15,'max':25},
               'B1c':{'min':15,'max':25},
                'F2':{'min':15,'max':25},
                'B2':{'min':15,'max':25},
               'F3':{'min':15,'max':25},
                'B3':{'min':15,'max':25},
                'FL':{'min':15,'max':25},
                'BL':{'min':15,'max':25}}


#generate primers class 
primers = LampPrimerGeneration(target_seq,length_rules)

In [ ]:
#general all primers and all sets 
all_primers, all_sets = primers.generate_tiling_sets(frame_size = 4)

In [ ]:
plt.title("length of primers")

vals = [] 
for ptype, ps in all_primers.items() :
    for primer, values in ps.items() :
        vals.append(len(values.get('sequence')))
        
plt.hist(vals)

In [ ]:
filtered_sets = primers.filter_possible_primers(my_filters = ['gc'])

In [ ]:
block_psets = primers.plot_sets_per_block(10, 350) 

In [ ]:
primers.downselect_and_limit_psets_per_transition_block(n_max_block = 50, n_max_transition = 5)

In [ ]:
primers.plot_amplicon_starts(bin_size = 20)

In [ ]:
primers.visualize_binding_sites(limit = 300)

## Primer bind and primer ordering files 
Primer bind file = F1c/F2 and B1c/B2 as separate sequences; this file is useful for testing binding to target sequences in geneious etc; format = csv
Primer ordering file = FIP and BIP as a single sequence; this file is for ordering primers from IDT ; format = excel 

In [ ]:
primer_bind_df, primer_order_df = primers.generate_primer_dfs()


In [ ]:
primer_bind_df.to_csv("mut_targ_testing/4-21_primer_bind.csv")

In [ ]:
#check if there are any duplicate primer sequences 
duplicated_primer_seqs = primer_order_df.loc[primer_order_df.sequence.duplicated()].sequence
print(f"Found {len(duplicated_primer_seqs)} duplicated sequences")

primer_order_df.loc[primer_order_df.sequence.isin(duplicated_primer_seqs)].sort_values(by = "sequence")

In [ ]:
len(set(primer_bind_df.primer_set)) / 21

In [ ]:
len(primer_order_df)

## plot characteristics 

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1,3,figsize = (12,4)) 

# 2.3 GC content
# Primers are designed so that their GC content is between about 40% to 65%.
# Primers with GC content between 50% and 60% tend to give relatively good primers. 

gcs = []
tms = []
dgs = []

for ptype, primers in all_primers.items() : 
    for primer, val in primers.items() : 
        gcs.append(gc_fraction(val.get('sequence')))
        tms.append(p3b.calc_tm(val.get('sequence')))
        dgs.append(p3b.calc_homodimer(val.get('sequence')).dg)
    

ax1.hist(gcs, alpha = 0.5, color = 'blue')
ax1.axvline(x = 0.4, linestyle = "--", color = "black")
ax1.axvline(x = 0.65, linestyle = "--", color = "black")
ax1.set_title("GC fraction for all primers") 
ax1.set_ylabel("Count")
ax1.set_xlabel("GC fraction")

ax2.hist(tms, alpha = 0.5, color = 'magenta')
ax2.set_title("Tm for all primers") 
ax2.set_ylabel("Count")
ax2.set_xlabel("Tm (C)")

ax3.hist(dgs, alpha = 0.5, color = 'seagreen')
ax3.axvline(x = -9000, linestyle = "--", color = "black")
ax3.set_title("Homodimer deltaG for all primers") 
ax3.set_ylabel("Count")
ax3.set_xlabel("dG")

In [ ]:
#plot tms for each primer type 
# The Tm for each region is designed to be about 65°C (64 - 66°C) for F1c and B1c, about
#60°C (59 - 61°C) for F2, B2, F3, and B3, and about
#65°C (64 - 66°C) for the loop primers. 
# (oligo concentration at 0.1 µM, sodium
#ion concentration at 50 mM, magnesium ion concentration at 4 mM).

tm_dict = {"F3":{'min':59,'max':61},"FL":{'min':64,'max':66},"F2":{'min':59,'max':61},"F1c":{'min':64,'max':66},
           "B3":{'min':59,'max':61},"BL":{'min':64,'max':66},"B2":{'min':59,'max':61},"B1c":{'min':64,'max':66}}

fig, axes = plt.subplots(2,4, figsize = (14,6), sharex = True)
fig.tight_layout(h_pad = 4, w_pad = 4) 

for ax, ptype in zip(axes.ravel(),["F3","FL","F2","F1c","B3","BL","B2","B1c"]) : 
    tms = []
    for primer, val in all_primers[ptype].items() : 
        tms.append(p3b.calc_tm(val.get('sequence'), mv_conc=50, dv_conc=4, dna_conc=100))
        
    ax.hist(tms)
    ax.set_title(f"Tm for all {ptype} primers") 
    ax.set_ylabel("Count")
    ax.set_xlabel("Tm (C)")
    ax.axvline(x = tm_dict[ptype].get('min'),linestyle = "--", color = "black")
    ax.axvline(x = tm_dict[ptype].get('max'),linestyle = "--", color = "black")
    
        

In [ ]:
#plot length dist


fig, axes = plt.subplots(2,4, figsize = (14,6), sharex = False)
fig.tight_layout(h_pad = 4, w_pad = 4) 

for ax, ptype in zip(axes.ravel(),["F3","FL","F2","F1c","B3","BL","B2","B1c"]) : 
    lens = []
    for primer, val in all_primers[ptype].items() : 
        lens.append(len(val.get('sequence')))
        
    ax.hist(lens, bins = np.arange( length_rules[ptype].get('min') - 1, length_rules[ptype].get('max') + 2,1))
    ax.set_title(f"Length for all {ptype} primers") 
    ax.set_ylabel("Count")
    ax.set_xlabel("Length (bp)")
    ax.axvline(x = length_rules[ptype].get('min'),linestyle = "--", color = "black")
    ax.axvline(x = length_rules[ptype].get('max'),linestyle = "--", color = "black")
    

### start position, Amplicon and loop lengths 

In [ ]:
#calculate amplicon characteristics 
#order = ["F3","F2","FL","F1c","B1c","BL","B2","B3"]

amp_lengths = []
F_loop_lengths = [] 
R_loop_lengths = []
for pset in all_sets: 
    F2 = pset[1]
    B2 = pset[6]
    F1c = pset[3]
    B1c = pset[4]
    
    amp_lengths.append(all_primers["B2"].get(B2).get('end') - all_primers["F2"].get(F2).get('start'))

    F_loop_lengths.append(all_primers["F1c"].get(F1c).get('start') - all_primers["F2"].get(F2).get('start'))
    
    R_loop_lengths.append(all_primers["B2"].get(B2).get('start') - all_primers["B1c"].get(B1c).get('start'))
    
#plot 
fig, ((ax1,ax2),(ax3,ax4)) = plt.subplots(2,2,figsize = (10,6)) 
fig.tight_layout(h_pad = 6, w_pad = 4)

ax1.hist([all_primers['F3'][primer].get('start') for primer in [pset[0] for pset in all_sets]],
     color = "orange",alpha = 0.7,bins = np.arange(0,len(target_seq) + 50,50))
ax1.set_title("F3 start position")


ax2.hist(amp_lengths, color = "blue", alpha = 0.7, bins = np.arange(120,170,5))
ax2.axvline(x = 120, linestyle = "--", color = "grey")
ax2.axvline(x = 160, linestyle = "--", color = "grey")
ax2.set_title("Amplicon length distribution") 

ax3.hist(F_loop_lengths, color = "green", alpha = 0.7,bins = np.arange(30,70,3))
ax3.axvline(x = 40, linestyle = "--", color = "grey")
ax3.axvline(x = 60, linestyle = "--", color = "grey") 
ax3.set_title("Forwared loop length distribution")

ax4.hist(R_loop_lengths, color = "red", alpha = 0.7,bins = np.arange(30,70,3))
ax4.axvline(x = 40, linestyle = "--", color = "grey")
ax4.axvline(x = 60, linestyle = "--", color = "grey") 
ax4.set_title("Reverse loop length distribution")


fig.suptitle("Amplicon characteristics: all primer sets", y = 1.1) 


## Visualize minimum within set delta G

In [ ]:
# Get rid of the first level of keys
flattened_dict = {}
for key, nested_dict in all_primers.items():
    flattened_dict.update(nested_dict)

flattened_dict[list(flattened_dict.keys())[0]]

#plot for 1000 
min_heterdimer_dg = []

# Adjusted loop
for pset in all_sets:  #
    
    dgs = []
    for pair in itertools.combinations(pset, 2):
        result = p3b.calc_heterodimer(flattened_dict[pair[0]].get('sequence'),flattened_dict[pair[1]].get('sequence'))
        dgs.append(result.dg)

    min_heterdimer_dg.append(np.min(dgs))

plt.hist(min_heterdimer_dg, color = "purple", alpha = 0.7)
minimum_dg = -9000

plt.axvline(x = minimum_dg, linestyle = "--", color = "darkgrey")

plt.title("Histogram: minimum deltaG for all possible dimers in primer set")
plt.ylabel("# of primer sets")
plt.xlabel("deltaG")

### Mismatches to targets

In [ ]:
#define a function to return the position and base for any discordant bases 

def count_discordant_positions(string1, string2):
    
    """ function to count discordant positions between string1 (here considered the reference) and string2 """ 
    # Ensure strings are of equal length
    if len(string1) != len(string2):
        return np.nan
    
    #check for degenerates 
    
    # Iterate over each character in the strings
    discordant_positions = 0
    for char1, char2 in zip(string1, string2):
        #check for degenerates 
        
        if char1 == char2 : 
            pass 
        
        else : 
            discordant_positions += 1 
        
    return discordant_positions

In [ ]:
if target_type == "Chan" :
    targets = list(pd.read_csv('target_sequences/1106_target_order_chan.csv').loc[:,"Sequence"])
    names = list(pd.read_csv('target_sequences/1106_target_order_chan.csv').loc[:,"Name"])
elif target_type == "ADAPT":
    target_df = pd.read_csv('target_sequences/ADAPT_library.targets.tsv', sep = "\t")
    names = list(target_df.loc[target_df.type == "exp","name"])
    targets =list(target_df.loc[target_df.type == "exp","target_seq"])

else :
    print("Error! Must select target type")

In [ ]:
ptypes = ["F3","F2","FL","F1c","B1c","BL","B2","B3"]
pset = random.choice(potential_sets)


dfs = []
for target,name in zip(targets,names) : 
    difs = []
    for primer,ptype in zip(pset,ptypes) :
        #get primer seqeunce 
        primer_seq = filtered_dict[ptype].get(primer).get("sequence")

        #get binding site sequences, depending on direction of primer  
        if filtered_dict[ptype].get(primer).get("direction") == "RC":
            binding_site = str(Seq(target[filtered_dict[ptype].get(primer).get("start"):filtered_dict[ptype].get(primer).get("end")]).reverse_complement())

        elif filtered_dict[ptype].get(primer).get("direction") == "F":
            binding_site = target[filtered_dict[ptype].get(primer).get("start"):filtered_dict[ptype].get(primer).get("end")]
        
        else: 
            print("Error! Invalid primer direction; must be RC or F")
            break

        difs.append(count_discordant_positions(binding_site,primer_seq))
    
    dfs.append(pd.DataFrame.from_dict(dict(zip(["F3","F2","FL","F1c","B1c","BL","B2","B3"],difs)),orient = 'index',columns= [name]))
    
fig, ax1 = plt.subplots(1,1, figsize = (16,3))

sns.heatmap(data =pd.concat(dfs,axis = 1), cmap = "Blues", ax =ax1)

fig.suptitle(f"Mismatches in each primer for primer set: {pset} \nand {len(targets)} targets", y = 1.1, fontsize = 10)

In [ ]:
plt.hist(pd.concat(dfs,axis = 1).sum(axis = 0), 
         bins = np.arange(0,np.max(pd.concat(dfs,axis = 1).sum(axis = 0))+1,1))

plt.title(f"Histogram of mismatches for whole primer set across all\nexperimental targets (n = {len(targets)})")
plt.ylabel("Number of primer sets")
plt.xlabel("Mismatches")

In [ ]:
pd.concat(dfs,axis = 1)['Target_WT_4']

# making order file from existing primer bind 

In [ ]:
import pandas as pd 
primers = pd.read_csv("/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/mut_targ_testing/4-21_primer_bind.csv")
primers.head()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(primers.loc[primers.primer_type == "F3"].start , primers.loc[primers.primer_type == "F3"].primer_set)

In [ ]:
import re 
def pad_primer_col(col):
    m = re.match(r'^primer-(\d+)$', col)
    if m:
        return f'primer-{int(m.group(1)):04d}'
    else:
        return col

primers['primer_name'] = primers["Unnamed: 0"].apply(lambda x: '-'.join(["LML",x.split("-")[1].zfill(4)]))
primers['primer_set'] = primers['primer_set'].apply(lambda x: "LML-PS" + str(x).zfill(4))
primers.head()

In [ ]:
primers.loc[primers.primer_set.isin(primer_sets[0:96])].to_csv(f"plate1_primer_bind_{date.today()}_v2.csv")

In [ ]:
plt.hist(primers.loc[primers.primer_set.isin(primer_sets[96:])].start)

In [ ]:
primer_seq_dict = dict(zip(primers.primer_name, primers.sequence))

In [ ]:
primers.loc[primers.primer_set == "LML-PS0000"][['primer_type','primer_name']]

In [ ]:
primer_sets = sorted(list(set(primers['primer_set'])))
primer_sets[0:10]

In [ ]:

pset_dict = {}
#make pset_dict 
for pset in primer_sets : 
    pset_rows = primers.loc[primers.primer_set == pset]
    pset_dict[pset] = dict(zip(pset_rows.primer_type, pset_rows.primer_name))

In [ ]:
from datetime import date
#pd.DataFrame.from_dict(pset_dict,orient = 'index').to_csv(f"LAMP_ML_pset_dict_{date.today()}.csv")

In [ ]:
primers = primers.set_index("primer_name").rename(columns = {'Unnamed : 0' : 'old_name'})

In [ ]:
import numpy as np
np.arange(0,len(primer_sets), 16)

In [ ]:
# I want to start with A1 - A6, then go to B1-B6, then C1-C6
plate_dfs = [] 

for i, start in enumerate(np.arange(0,len(primer_sets), 16)) : 
    plate_psets = primer_sets[start:start+16]

    nrows = {}

    for pset, well_let in zip(plate_psets[:8],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL','BL'],range(4)):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}05"] = {"Name" : FIP}

        nrows[f"{well_let}05"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}05"]["primer_type"] = "FIP" 
        nrows[f"{well_let}05"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}06"] = {"Name" : BIP}

        nrows[f"{well_let}06"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}06"]["primer_type"] = "BIP" 
        nrows[f"{well_let}06"]["primer_set"] = pset

    for pset, well_let in zip(plate_psets[8:],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL'],[6,7,8]):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        for ptype, well_num in zip(['BL'],[9]):

            nrows[f"{well_let}{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}11"] = {"Name" : FIP}

        nrows[f"{well_let}11"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}11"]["primer_type"] = "FIP" 
        nrows[f"{well_let}11"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}12"] = {"Name" : BIP}

        nrows[f"{well_let}12"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}12"]["primer_type"] = "BIP" 
        nrows[f"{well_let}12"]["primer_set"] = pset

    plate_dfs.append(pd.DataFrame.from_dict(nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
                        {'sequence':'Sequence'}))

In [ ]:
for i, start in enumerate(np.arange(16,len(primer_sets), 44)) : 
    print(primer_sets[start])

In [ ]:
# I want to start with A1 - A6, then go to B1-B6, then C1-C6
plate_dfs = [] 
partial_plate_dfs = []
for i, start in enumerate(np.arange(16,len(primer_sets), 44)) : 
    tri_plate_psets = primer_sets[start:start+44]

    nrows = {}

    for pset, well_let in zip(tri_plate_psets[:8],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL','BL'],range(4)):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}05"] = {"Name" : FIP}

        nrows[f"{well_let}05"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}05"]["primer_type"] = "FIP" 
        nrows[f"{well_let}05"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}06"] = {"Name" : BIP}

        nrows[f"{well_let}06"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}06"]["primer_type"] = "BIP" 
        nrows[f"{well_let}06"]["primer_set"] = pset

    for pset, well_let in zip(tri_plate_psets[8:16],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL'],[6,7,8]):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        for ptype, well_num in zip(['BL'],[9]):

            nrows[f"{well_let}{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}11"] = {"Name" : FIP}

        nrows[f"{well_let}11"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}11"]["primer_type"] = "FIP" 
        nrows[f"{well_let}11"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}12"] = {"Name" : BIP}

        nrows[f"{well_let}12"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}12"]["primer_type"] = "BIP" 
        nrows[f"{well_let}12"]["primer_set"] = pset
        
    plate_dfs.append(pd.DataFrame.from_dict(nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
                        {'sequence':'Sequence'}))
        
    # now do the partial plate LEFT 
    part_nrows = {} 
    
    for pset, well_let in zip(tri_plate_psets[16:22],["A","B","C","D","E","F"]) : 
        for ptype, well_num in zip(['F3','B3','FL','BL'],range(4)):

            part_nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            part_nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            part_nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        part_nrows[f"{well_let}05"] = {"Name" : FIP}

        part_nrows[f"{well_let}05"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        part_nrows[f"{well_let}05"]["primer_type"] = "FIP" 
        part_nrows[f"{well_let}05"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        part_nrows[f"{well_let}06"] = {"Name" : BIP}

        part_nrows[f"{well_let}06"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        part_nrows[f"{well_let}06"]["primer_type"] = "BIP" 
        part_nrows[f"{well_let}06"]["primer_set"] = pset


#     partial_plate_dfs.append(pd.DataFrame.from_dict(part_nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
#                         {'sequence':'Sequence'}))
    
    #now do full plate 2
    nrows = {}

    for pset, well_let in zip(tri_plate_psets[22:30],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL','BL'],range(4)):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}05"] = {"Name" : FIP}

        nrows[f"{well_let}05"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}05"]["primer_type"] = "FIP" 
        nrows[f"{well_let}05"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}06"] = {"Name" : BIP}

        nrows[f"{well_let}06"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}06"]["primer_type"] = "BIP" 
        nrows[f"{well_let}06"]["primer_set"] = pset

    for pset, well_let in zip(tri_plate_psets[30:38],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL'],[6,7,8]):

            nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        for ptype, well_num in zip(['BL'],[9]):

            nrows[f"{well_let}{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            nrows[f"{well_let}{well_num+1}"]["primer_set"] = pset
            nrows[f"{well_let}{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        nrows[f"{well_let}11"] = {"Name" : FIP}

        nrows[f"{well_let}11"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        nrows[f"{well_let}11"]["primer_type"] = "FIP" 
        nrows[f"{well_let}11"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        nrows[f"{well_let}12"] = {"Name" : BIP}

        nrows[f"{well_let}12"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        nrows[f"{well_let}12"]["primer_type"] = "BIP" 
        nrows[f"{well_let}12"]["primer_set"] = pset
        
    plate_dfs.append(pd.DataFrame.from_dict(nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
                        {'sequence':'Sequence'}))
        
    # now do the partial plate RIGHT
    
    for pset, well_let in zip(tri_plate_psets[38:44],["A","B","C","D","E","F","G","H"]) : 
        for ptype, well_num in zip(['F3','B3','FL'],[6,7,8]):

            part_nrows[f"{well_let}0{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            part_nrows[f"{well_let}0{well_num+1}"]["primer_set"] = pset
            part_nrows[f"{well_let}0{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        for ptype, well_num in zip(['BL'],[9]):

            part_nrows[f"{well_let}{well_num+1}"] = dict(primers.loc[pset_dict[pset].get(ptype)])
            part_nrows[f"{well_let}{well_num+1}"]["primer_set"] = pset
            part_nrows[f"{well_let}{well_num+1}"]["Name"] = pset_dict[pset].get(ptype)

        FIP = f"{pset_dict[pset].get('F1c')}-{pset_dict[pset].get('F2')}"

        part_nrows[f"{well_let}11"] = {"Name" : FIP}

        part_nrows[f"{well_let}11"]["sequence"] = primers.loc[pset_dict[pset].get('F1c'),"sequence"] + primers.loc[pset_dict[pset].get('F2'),"sequence"]
        part_nrows[f"{well_let}11"]["primer_type"] = "FIP" 
        part_nrows[f"{well_let}11"]["primer_set"] = pset


        BIP = f"{pset_dict[pset].get('B1c')}-{pset_dict[pset].get('B2')}"

        part_nrows[f"{well_let}12"] = {"Name" : BIP}

        part_nrows[f"{well_let}12"]["sequence"] = primers.loc[pset_dict[pset].get('B1c'),"sequence"] + primers.loc[pset_dict[pset].get('B2'),"sequence"]
        part_nrows[f"{well_let}12"]["primer_type"] = "BIP" 
        part_nrows[f"{well_let}12"]["primer_set"] = pset

    plate_dfs.append(pd.DataFrame.from_dict(part_nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
                        {'sequence':'Sequence'}))

In [ ]:
len(plate_dfs[0:26])

In [ ]:
tdf = plate_dfs[24][["Well Position","Name","Sequence","primer_set"]].copy()

tdf['Row'] = tdf['Well Position'].str[0]
tdf['Col'] = tdf['Well Position'].str[1:].str.zfill(2)  # ensures e.g. 1 → 01

# Pivot such that Rows = 'Row', Columns = 'Col', values='Name'
tdf.pivot(index='Row', columns='Col', values='primer_set')

In [ ]:
tdf = plate_dfs[25][["Well Position","Name","Sequence","primer_set"]].copy()

tdf['Row'] = tdf['Well Position'].str[0]
tdf['Col'] = tdf['Well Position'].str[1:].str.zfill(2)  # ensures e.g. 1 → 01

# Pivot such that Rows = 'Row', Columns = 'Col', values='Name'
tdf.pivot(index='Row', columns='Col', values='primer_set')

In [ ]:
tdf = plate_dfs[27][["Well Position","Name","Sequence","primer_set"]].copy()

tdf['Row'] = tdf['Well Position'].str[0]
tdf['Col'] = tdf['Well Position'].str[1:].str.zfill(2)  # ensures e.g. 1 → 01

# Pivot such that Rows = 'Row', Columns = 'Col', values='Name'
tdf.pivot(index='Row', columns='Col', values='Name')

In [ ]:
out = pd.DataFrame.from_dict(nrows,orient = 'index').to_csv(f"plate1_orders_{date.today()}_v2.csv")

out

In [ ]:
order_df = pd.DataFrame.from_dict(nrows,orient = 'index').reset_index(names = ['Well Position']).rename(columns = 
                        {'sequence':'Sequence'})

order_df[["Well Position","Name","Sequence"]]#.to_excel(f"final_primer_order_6-05/plate1_order_{date.today()}_v2.xlsx")


In [ ]:
plate_dfs[28]

In [ ]:

with pd.ExcelWriter(f'final_primer_order_6-05/reodered_plates11-29_{date.today()}.xlsx') as writer:
    for i, dataframe in enumerate(plate_dfs[9:28]):
        sheetname = f"LML_plate{i+11}"
        dataframe[["Well Position","Name","Sequence","primer_set"]].to_excel(writer, sheet_name=sheetname, index=False)

In [ ]:
# Replace with your Excel filename
fname = f'final_primer_order_6-05/reodered_all_plates_{date.today()}.xlsx'

# Read all sheets into a dictionary of DataFrames
tdfs = pd.read_excel(fname, sheet_name=None)

# Extract the "Name" column from each sheet (if present) and flatten into a list
names = []
for tsdf in tdfs.values():
    if 'primer_set' in tdf.columns:
        names.extend(tsdf['primer_set'].dropna().tolist())

# Now 'names' is a flat list of all values in the 'Name' column from all sheets
print(len(names))
print(len(set(names)))

In [ ]:
plate_dfs[0]

In [ ]:
for i, dataframe in enumerate(plate_dfs[:28]):
    primer_sets_in_plate = set(dataframe.primer_set)
    primers.loc[primers.primer_set.isin(primer_sets_in_plate)].to_csv(f"final_primer_order_6-05/primer_binds/plate{i+2}_primer_bind_{date.today()}.csv")